# Main Notebook

#### Imports and Plot Setup

In [1]:
# Imports
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
import json
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

#### Github Authorization

In [2]:
from config import GITHUB_TOKEN

headers = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

BASE_URL = "https://api.github.com"

#### Check for secrets leak

In [3]:
import re

def check_for_secrets(notebook_path):
    """Check if notebook contains potential secrets"""
    with open(notebook_path, 'r') as f:
        content = f.read()
    
    patterns = [
        r'ghp_[a-zA-Z0-9]{36}',  # GitHub personal access token
        r'github_pat_[a-zA-Z0-9]{22}_[a-zA-Z0-9]{59}',  # GitHub fine-grained token
        r'GITHUB_TOKEN\s*=\s*["\'][^"\']+["\']',  # Hardcoded token assignment
    ]
    
    for pattern in patterns:
        if re.search(pattern, content):
            print("WARNING: Potential secret found!")
            return False
    
    print("No obvious secrets detected")
    return True

check_for_secrets('notebook.ipynb')

No obvious secrets detected


True

#### Function to check Rate limits

In [4]:
def check_rate_limit():
    """Check remaining API calls"""
    response = requests.get(f"{BASE_URL}/rate_limit", headers=headers)
    data = response.json()
    return data['rate']['remaining'], data['rate']['reset']

check_rate_limit()

(4977, 1773416861)

#### Data Collection Functions

In [ ]:
def search_repositories(query, sort='stars', order='desc', per_page=100, max_pages=10):
    """
    Search GitHub repositories
    
    Parameters:
    - query: search query (e.g., 'stars:>1000', 'language:python')
    - sort: 'stars', 'forks', 'updated'
    - order: 'asc' or 'desc'
    """
    repos = []
    
    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/search/repositories"
        params = {
            'q': query,
            'sort': sort,
            'order': order,
            'per_page': per_page,
            'page': page
        }
        
        response = requests.get(url, headers=headers, params=params)
        
        if response.status_code == 200:
            data = response.json()
            repos.extend(data['items'])
            
            # Check if there are more pages
            if len(data['items']) < per_page:
                break
                
            # Respect rate limiting
            time.sleep(1)
        else:
            print(f"Error: {response.status_code}")
            break
    
    return repos



#################################################################

def get_repo_languages(owner, repo_name):
    """Get languages used in a repository"""
    url = f"{BASE_URL}/repos/{owner}/{repo_name}/languages"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    return {}



#################################################################

def get_repo_topics(owner, repo_name):
    """Get topics/tags for a repository"""
    url = f"{BASE_URL}/repos/{owner}/{repo_name}/topics"
    headers_topics = headers.copy()
    headers_topics['Accept'] = 'application/vnd.github.mercy-preview+json'
    
    response = requests.get(url, headers=headers_topics)
    
    if response.status_code == 200:
        return response.json().get('names', [])
    return []


################################################################

print("Functions defined successfully!")

#### Collect Repository Data

In [6]:
print("Collecting top repositories...")

# Collect top starred repos
top_repos = search_repositories(
    query='stars:>5000',  # Adjust threshold as needed
    sort='stars',
    per_page=100,
    max_pages=10  # This gives you up to 1000 repos
)

print(f"Collected {len(top_repos)} repositories")

# Check rate limit
remaining, reset_time = check_rate_limit()
print(f"Rate limit remaining: {remaining}")

Collected 1000 repositories
Rate limit remaining: 4977
